<a href="https://colab.research.google.com/github/joblingjack/Visual-Analytics/blob/main/NC_Counties_1790_2020.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import json
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display, HTML

In [15]:
#read file which contains geo population data for 100 NC counties from 1790-2020
census = gpd.read_file("historic-census.geojson")
#filter for counties in geo dataframe
counties = census[census["areatype"] == "County"].copy()
# convert year from text to integer
counties["year"] = counties["year"].astype(int)

In [16]:
#display variables in the file
print(census.columns)

Index(['areatype', 'areaname', 'year', 'population', 'notes', 'geom',
       'centroid', 'geometry'],
      dtype='object')


In [17]:
counties["year"].unique()[:20]

array([1880, 1950, 1980, 1890, 2010, 1960, 1940, 1810, 1870, 1840, 1910,
       2000, 1800, 2020, 1920, 1860, 1900, 1970, 1830, 1990])

In [18]:
#calculate percent change over each year
counties = counties.sort_values(["areaname", "year"])

counties["PrevPop"] = (
    counties.groupby("areaname")["population"]
       .shift(1)
)

counties["PctChange"] = (
    (counties["population"] - counties["PrevPop"])
    / counties["PrevPop"]
) * 100

In [19]:
# Create dynamic color scale limits
max_change_overall = max(
    abs(counties['PctChange'].min(skipna=True) if not counties['PctChange'].isnull().all() else 0),
    abs(counties['PctChange'].max(skipna=True) if not counties['PctChange'].isnull().all() else 0)
)

# If max_change_overall is 0 provide a default to avoid issues
if max_change_overall == 0:
    max_change_overall = 1 # A small default value to avoid division by zero or an empty range

colormap = cm.LinearColormap(
    colors=["red", "white", "green"],
    vmin=-max_change_overall,
    vmax=max_change_overall
)
colormap.caption = "Population Change (%)"

# Define an output widget to display the map
map_output = widgets.Output()

def create_static_map_with_dropdown(selected_year):
    # Filter data for the selected year
    yearly_data = counties[counties["year"] == selected_year].copy()

    # Initialize the map, centered on North Carolina
    m = folium.Map(
        location=[35.5, -79.5],  # Approximate center of NC
        zoom_start=7             # Zoom level to show all of NC
    )

    # Define the style function for GeoJson features
    def style_function(feature):
        pct_change = feature["properties"]["PctChange"]
        if pd.isna(pct_change):
            fill_color = "#D3D3D3"  # Light grey for NaN values
        else:
            fill_color = colormap(pct_change)

        return {
            "fillColor": fill_color,
            "color": "black",
            "weight": 0.5,
            "fillOpacity": 0.8,
        }

    # Add GeoJson layer to the map
    folium.GeoJson(
        yearly_data, # Pass GeoDataFrame directly here
        name=f"NC Population Change {selected_year}",
        style_function=style_function,
        tooltip=folium.GeoJsonTooltip(
            fields=[
                "areaname",
                "population",
                "PctChange"
            ],
            aliases=[
                "County:",
                "Population:",
                "Population Change (%):"
            ],
            localize=True,
            sticky=False,
            labels=True,
            style="""
                background-color: #F0EFEF;
                color: #333333;
                font-family: sans-serif;
                font-size: 12px;
                padding: 10px;
            """
        )
    ).add_to(m)

    # Add the colormap legend to the map
    colormap.add_to(m)

    # Clear previous output and display new map within the output widget
    with map_output:
      map_output.clear_output(wait=True)
      display(HTML(m._repr_html_())) # Directly display the Folium map object, removed IFrame usage

# Get unique sorted years for the dropdown
unique_years = sorted(counties["year"].unique())

# Create the dropdown
year_dropdown = widgets.Dropdown(
    options=unique_years,
    value=unique_years[-1], # Set initial value to the latest year
    description='Select Year:',
    disabled=False,
)

# Link dropdown to the map generation function
year_dropdown.observe(lambda change: create_static_map_with_dropdown(change['new']), names='value')

# Display dropdown and the map output area
display(year_dropdown, map_output)

# Map display for the selected year
create_static_map_with_dropdown(year_dropdown.value)


Dropdown(description='Select Year:', index=23, options=(np.int64(1790), np.int64(1800), np.int64(1810), np.int…

Output()